03 - Offline inference -> submission.csv
#
# Attach: the competition, the code dataset, and every train-fold kernel.
# No internet is used, so this is valid as a code-competition submission.
#
# The hidden test set is preprocessed here from raw DICOM by the same code
# path training used, so there is no train/serve skew. Checkpoints carry the
# resolution they were trained at, so models from different corpora blend.

In [ ]:
import sys, subprocess
from pathlib import Path

CODE = next(
    (p for pat in ("rsna-knee-code", "*/rsna-knee-code", "*/*/rsna-knee-code")
     for p in sorted(Path("/kaggle/input").glob(pat)) if (p / "src").is_dir()),
    None,
)
assert CODE is not None, "rsna-knee-code dataset is not attached to this notebook"
sys.path.insert(0, str(CODE / "src"))
sys.path.insert(0, str(CODE))
print("CODE", CODE)

# timm ships in the Kaggle image; the install is a fallback for images that lack
# it, and cannot run in the offline submission notebook.
try:
    import timm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=False)
    import timm

import rsna_knee.config as cfg
print("RAW ", cfg.RAW, cfg.RAW.exists())
print("PREP", cfg.PREP, cfg.PREP.exists())
print("SIZE", cfg.SIZE, cfg.MOSAIC_EXT)

In [ ]:
from pathlib import Path
# Checkpoints arrive from two places with different shapes: a training kernel's
# output keeps its models/ directory, while a dataset upload has the directory
# flattened away by Kaggle. So locate the *inputs* whose name says they hold
# weights, then search inside them, rather than assuming a layout -- a glob that
# required models/ silently found nothing and failed the submission.
ROOTS = [p for pat in ("*", "*/*", "*/*/*")
         for p in Path("/kaggle/input").glob(pat)
         if p.is_dir() and "rsna-knee-train" in p.name]
CKPTS = sorted({str(q) for r in ROOTS for q in r.rglob("*.pt")})
assert CKPTS, f"no checkpoints found under {[str(r) for r in ROOTS]}"
for c in CKPTS:
    print(c)
# A shell magic interpolates a bare {name}; build the argument string here rather
# than relying on an expression inside the braces.
CKPT_ARGS = " ".join(CKPTS)

In [ ]:
!cd {CODE} && python scripts/infer.py --split test --out /kaggle/working/submission.csv \
    --ckpt {CKPT_ARGS}

In [ ]:
import pandas as pd
sub = pd.read_csv("/kaggle/working/submission.csv")
print(sub.shape)
sub.head()